# Notebook 02: Sensitivity Analysis
**Run after:** Notebook 01

Grid: β ∈ {0.00, 0.05, 0.10, 0.15, 0.20, 0.25} × n ∈ {500, 1000, 5000, 50000} × 20 seeds

**Estimated runtime:**
- With `n_jobs=1` (serial): ~45-90 minutes
- With `n_jobs=-1` (all Colab cores, usually 2): ~25-45 minutes
- With Colab Pro (more cores): ~10-20 minutes

**Tip:** Enable GPU runtime in Colab (Runtime > Change runtime type) — won't speed up
these algorithms directly, but gives you more CPU cores via the high-RAM option.

In [ ]:
!pip install -q causal-learn dowhy networkx matplotlib numpy pandas scipy scikit-learn joblib tqdm

In [ ]:
import os, sys
# Mount Google Drive or clone repo (see Notebook 01 Step 2)
# Then set this to your project root:
PROJECT_ROOT = '/content/causal_bias_project'   # <-- adjust if needed
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)
print('Working directory:', os.getcwd())

## Check available CPU cores

In [ ]:
import multiprocessing
n_cores = multiprocessing.cpu_count()
print(f'Available CPU cores: {n_cores}')
print(f'Recommended n_jobs: {n_cores}')

## Run the sensitivity grid
**Set `n_jobs` below.** Use `n_jobs=-1` to use all available cores.
Set `n_jobs=1` if you get pickling errors (some Colab environments don't support multiprocessing).

In [ ]:
import importlib
import synthetic.sensitivity_analysis as sens
importlib.reload(sens)

N_JOBS = -1   # ← change to 1 if multiprocessing fails

df_sens = sens.run_sensitivity(n_jobs=N_JOBS)
print(f'\nComplete. Total records: {len(df_sens)}')
print(df_sens.groupby('algorithm')['detected_biased'].mean().round(3))

## Quick summary: detection rates by algorithm and n

In [ ]:
import pandas as pd
import numpy as np

df_sens = pd.read_pickle('results/sensitivity_results.pkl')

# Detection rate pivot: rows=n, cols=algorithm, values=mean detection at beta=0.15
sub = df_sens[df_sens['beta'] == 0.15]
pivot = sub.groupby(['n','algorithm'])['detected_biased'].mean().unstack()
pivot.columns.name = None
pivot.index.name = 'n'
print('Detection rate at β=0.15 (0.0 = never detected, 1.0 = always detected):')
print(pivot.round(2).to_string())

print('\nFalse positive rate at β=0.0 (lower is better):')
sub0 = df_sens[df_sens['beta'] == 0.00]
fp = sub0.groupby(['n','algorithm'])['detected_biased'].mean().unstack()
fp.columns.name = None
print(fp.round(2).to_string())

## LiNGAM coefficient separation across conditions

In [ ]:
# Show how β̂ separates biased vs unbiased at different n
for alg in ['ICA-LiNGAM','DirectLiNGAM']:
    print(f'\n{alg} — β̂ for Race→Loan at β=0.15 (biased) and β=0.00 (unbiased):')
    for n in [500, 1000, 5000, 50000]:
        sub = df_sens[(df_sens['algorithm']==alg) & (df_sens['n']==n)]
        b_hat_biased   = sub[sub['beta']==0.15]['beta_hat_biased'].mean()
        b_hat_unbiased = sub[sub['beta']==0.00]['beta_hat_unbiased'].mean()
        gap = abs(b_hat_biased - b_hat_unbiased)
        print(f'  n={n:6d}: biased={b_hat_biased:+.4f}  '
              f'unbiased={b_hat_unbiased:+.4f}  gap={gap:.4f}')

## Generate Figure 2 (sensitivity heatmap)

In [ ]:
import importlib, figures.fig_sensitivity as f2
importlib.reload(f2)
os.makedirs('figures/output', exist_ok=True)
f2.main()

In [ ]:
from IPython.display import Image
Image('figures/output/fig_sensitivity.png')